# Logica para obtener el score basado en la matriz de riesgo

## importar las libreras

In [95]:
!pip install pymongo

In [96]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
import json
from bson import ObjectId
from datetime import datetime, timezone

In [97]:
load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DATABASE_NAME = os.getenv("DATABASE_NAME", "sistema_prevencion")

client = MongoClient(MONGO_URI)
db = client[DATABASE_NAME]

In [99]:
dataframes = {}
for nombre in colecciones:
    dataframes[nombre] = pd.DataFrame(list(db[nombre].find()))
    print(f"✓ {nombre}: {len(dataframes[nombre])} documentos")

# Acceder a cada una
cat_paises             = dataframes["paises"]
cat_nacionalidad       = dataframes["nacionalidad"]
cat_divisa             = dataframes["divisa"]
cat_actividad_economica = dataframes["actividad-economica"]
cat_tipo_persona         = dataframes["tipo-persona"]
cat_origen_recursos     = dataframes["origen-recursos"]
car_destino_recursos     = dataframes["detino-recursos"]
cat_producto_servicio     = dataframes["producto-servicio-sencible"]
cat_entidad_federativa = dataframes["entidad-federativa"]

✓ actividad-vulnerable: 0 documentos
✓ divisa: 4 documentos
✓ tipo-persona: 5 documentos
✓ nacionalidad: 249 documentos
✓ entidad-federativa: 33 documentos
✓ producto-servicio-sencible: 13 documentos
✓ uso-lendero-pay: 0 documentos
✓ detino-recursos: 9 documentos
✓ grado-estudios: 0 documentos
✓ entidad-financiera-vulnerable: 0 documentos
✓ origen-recursos: 21 documentos
✓ paises: 3 documentos
✓ actividad-economica: 467 documentos


In [100]:
## Ponderaciones según la matríz de riesgo
PONDERACIONES = {
    "pais_nacionalidad":     0.05,
    "pais_residencia":       0.08,
    "pais_origen_recursos":  0.08,
    "pais_destino_recursos": 0.08,
    "nacionalidad":          0.08,
    "actividad_economica":   0.08,
    "tipo_persona":          0.05,
    "origen_recursos":       0.04,
    "destino_recursos":      0.02,
    "producto_servicio":     0.05,
    "entidad_federativa":    0.08,  # ajusta si aplica
}

In [102]:
# Crear diccionarios de mapeo {clave: score} desde cada catálogo
map_paises           = dict(zip(cat_paises["clave_pais"],            cat_paises["score_pais"]))
map_paises_residencia           = dict(zip(cat_paises["clave_pais"],            cat_paises["score_presidencia"]))
map_paises_origen_recursos           = dict(zip(cat_paises["clave_pais"],            cat_paises["score_pais_origen_recursos"]))
map_paises_destino_recursos           = dict(zip(cat_paises["clave_pais"],            cat_paises["score_pais_destino_recursos"]))

map_nacionalidad     = dict(zip(cat_nacionalidad["clave_nacionalidad"],      cat_nacionalidad["score_nacionalidad"]))
map_actividad = {
    (str(row["clave_tipopersona"]), str(row["clave_acteconomica"])): row["score_acteconomica"]
    for _, row in cat_actividad_economica.iterrows()
}

map_tipo_persona = dict(zip(cat_tipo_persona["clave_tipopersona"], cat_tipo_persona["score_tper"]))
map_origen_recursos  = dict(zip(cat_origen_recursos["clave_origrecursos"],   cat_origen_recursos["score_origrecursos"]))
map_destino_recursos = dict(zip(car_destino_recursos["clave_desrecursos"],  car_destino_recursos["score_desrecursos"]))
map_producto         = dict(zip(cat_producto_servicio["clave_prodser"], cat_producto_servicio["score_prodser"]))
map_entidad          = dict(zip(cat_entidad_federativa["clave_entidadfederativa"], cat_entidad_federativa["score_entidadfederativa"]))

In [ ]:
SCORE_MAXIMO = 5

# ── Tipo Persona ──────────────────────────────────────────────────────────────
def calcular_score_tipo_persona(df):
    df["score_tipo_persona"] = (
        df["clave_tipopersona"]
        .map(map_tipo_persona)
        .fillna(0)
        .mul(PONDERACIONES["tipo_persona"])
        .div(SCORE_MAXIMO)
    )
    return df["score_tipo_persona"]


# ── Actividad Económica (llave compuesta) ─────────────────────────────────────
def calcular_score_actividad_economica(df):
    df["score_actividad_economica"] = (
        df.apply(
            lambda row: map_actividad.get(
                (str(row["clave_tipopersona"]), str(row["clave_acteconomica"])),
                0  # default si no hay match
            ), axis=1
        )
        .mul(PONDERACIONES["actividad_economica"])
        .div(SCORE_MAXIMO)
    )
    return df["score_actividad_economica"]


# ── País Nacionalidad ─────────────────────────────────────────────────────────
def calcular_score_pais_nacionalidad(df):
    df["score_pais_nacionalidad"] = (
        df["clave_pais_nacionalidad"]
        .map(map_paises)
        .fillna(0)
        .mul(PONDERACIONES["pais_nacionalidad"])
        .div(SCORE_MAXIMO)
    )
    return df["score_pais_nacionalidad"]


# ── País Residencia ───────────────────────────────────────────────────────────
def calcular_score_pais_residencia(df):
    df["score_pais_residencia"] = (
        df["clave_pais_residencia"]
        .map(map_paises_residencia)
        .fillna(0)
        .mul(PONDERACIONES["pais_residencia"])
        .div(SCORE_MAXIMO)
    )
    return df["score_pais_residencia"]


# ── País Origen Recursos ──────────────────────────────────────────────────────
def calcular_score_pais_origen_recursos(df):
    df["score_pais_origen_recursos"] = (
        df["clave_pais_origen_recursos"]
        .map(map_paises_origen_recursos)
        .fillna(0)
        .mul(PONDERACIONES["pais_origen_recursos"])
        .div(SCORE_MAXIMO)
    )
    return df["score_pais_origen_recursos"]


# ── País Destino Recursos ─────────────────────────────────────────────────────
def calcular_score_pais_destino_recursos(df):
    df["score_pais_destino_recursos"] = (
        df["clave_pais_destino_recursos"]
        .map(map_paises_destino_recursos)
        .fillna(0)
        .mul(PONDERACIONES["pais_destino_recursos"])
        .div(SCORE_MAXIMO)
    )
    return df["score_pais_destino_recursos"]


# ── Nacionalidad ──────────────────────────────────────────────────────────────
def calcular_score_nacionalidad(df):
    df["score_nacionalidad"] = (
        df["clave_nacionalidad"]
        .map(map_nacionalidad)
        .fillna(0)
        .mul(PONDERACIONES["nacionalidad"])
        .div(SCORE_MAXIMO)
    )
    return df["score_nacionalidad"]


# ── Origen Recursos ───────────────────────────────────────────────────────────
def calcular_score_origen_recursos(df):
    df["score_origen_recursos"] = (
        df["clave_origrecursos"]
        .map(map_origen_recursos)
        .fillna(0)
        .mul(PONDERACIONES["origen_recursos"])
        .div(SCORE_MAXIMO)
    )
    return df["score_origen_recursos"]


# ── Destino Recursos ──────────────────────────────────────────────────────────
def calcular_score_destino_recursos(df):
    df["score_destino_recursos"] = (
        df["clave_desrecursos"]
        .map(map_destino_recursos)
        .fillna(0)
        .mul(PONDERACIONES["destino_recursos"])
        .div(SCORE_MAXIMO)
    )
    return df["score_destino_recursos"]


# ── Producto / Servicio ───────────────────────────────────────────────────────
def calcular_score_producto_servicio(df):
    df["score_producto_servicio"] = (
        df["clave_prodser"]
        .map(map_producto)
        .fillna(0)
        .mul(PONDERACIONES["producto_servicio"])
        .div(SCORE_MAXIMO)
    )
    return df["score_producto_servicio"]


# ── Entidad Federativa ────────────────────────────────────────────────────────
def calcular_score_entidad_federativa(df):
    df["score_entidad_federativa"] = (
        df["clave_entidadfederativa"]
        .map(map_entidad)
        .fillna(0)
        .mul(PONDERACIONES["entidad_federativa"])
        .div(SCORE_MAXIMO)
    )
    return df["score_entidad_federativa"]


def score_edad_constitucion(df,clave_tper,feca_constitucion):
    fecha_constitucion = pd.to_datetime(feca_constitucion, errors='coerce')
    fecha_actual = pd.to_datetime("today")
    edad_constitucion = (fecha_actual - fecha_constitucion).dt.days / 365.25

    if df[clave_tper].isin(1,2):
        if edad_constitucion < 18:
            return 0
        elif edad_constitucion < 24:
            return 3
        elif edad_constitucion < 36:
            return 1
        elif edad_constitucion < 46:
            return 1
        elif edad_constitucion < 61:
            return 1
        elif edad_constitucion > 60:
            return 2
        else:
            return None
        
    if df[clave_tper].isin(3):
        if edad_constitucion < 1:
            return 0
        elif edad_constitucion < 3:
            return 4
        elif edad_constitucion < 11:
            return 2
        elif edad_constitucion > 11:
            return 1
        else:
            return None
        
# Condición PEP
dataframe_onboarding["score_pep"] = np.where(dataframe_onboarding['PEP'] == True, 0.12,0.024) #-> Si es PEP asignar el score ya transformado según la ponderación, si no es PEP asignar el score ya transformado según la ponderación para no PEP
# ── Orquestadora ──────────────────────────────────────────────────────────────
def calcular_score_total(df):
    calcular_score_tipo_persona(df)
    calcular_score_actividad_economica(df)   # usa llave compuesta
    calcular_score_pais_nacionalidad(df)
    calcular_score_pais_residencia(df)
    calcular_score_pais_origen_recursos(df)
    calcular_score_pais_destino_recursos(df)
    calcular_score_nacionalidad(df)
    calcular_score_origen_recursos(df)
    calcular_score_destino_recursos(df)
    calcular_score_producto_servicio(df)
    calcular_score_entidad_federativa(df)

    cols_score = [col for col in df.columns if col.startswith("score_")]
    df["score_total"] = df[cols_score].sum(axis=1)

    return df